# Stack Overflow 2025 Survey - Community Platform Effectiveness Analysis

## Research Question
Which community platforms (Stack Overflow, Discord, Reddit, Dev.to, GitHub, etc.) do developers use at different career stages, and how does platform choice correlate with Stack Overflow engagement?

## Analysis Components
1. Cluster Analysis - Segment developers by community platform usage patterns
2. Cohort Analysis - Community platform usage by experience level
3. Platform Cluster vs SO Engagement Analysis
4. Summary and Key Insights

## Schema References
- SOVisitFreq (QID100) - Stack Overflow visit frequency
- SOPartFreq (QID102) - Stack Overflow participation frequency
- SOComm (QID106) - Stack Overflow community membership
- YearsCodePro (QID34) - Professional coding experience
- CommPlatformHaveWorkedWith - Community platforms used (Stack Overflow, Discord, Reddit, Dev.to, etc.)
- CommPlatformWantToWorkWith - Community platforms want to use
- CommPlatformAdmired - Community platforms admired

Author: Analysis Team  
Date: 2025

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import sys

# Machine Learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Statistical analysis
from scipy import stats
from scipy.stats import chi2_contingency

# Visualization
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except ImportError:
    print("Installing plotly for alluvial diagrams...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'plotly'])
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True

# Styling
plt.style.use('default')
sns.set_palette('husl')
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


In [3]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def safe_save_csv(df, filepath):
    """Safely save DataFrame to CSV with error handling"""
    try:
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        df.to_csv(filepath, index=False)
        print(f"   ✓ Saved to: {filepath}")
        return True
    except Exception as e:
        print(f"   ⚠ Error saving {filepath}: {str(e)}")
        return False

def create_experience_category(years):
    """Categorize developers by experience level"""
    if pd.isna(years):
        return 'Unknown'
    elif years < 2:
        return 'Junior (0-2 years)'
    elif years < 5:
        return 'Mid (2-5 years)'
    elif years < 10:
        return 'Senior (5-10 years)'
    else:
        return 'Expert (10+ years)'

def categorize_so_engagement(row):
    """Categorize Stack Overflow engagement level"""
    visit = str(row.get('SOVisitFreq', '')).lower()
    part = str(row.get('SOPartFreq', '')).lower()
    
    if 'multiple times per day' in visit or 'daily' in visit:
        if 'multiple times per week' in part or 'daily' in part:
            return 'Very Active'
        elif 'few times per month' in part or 'weekly' in part:
            return 'Active'
        else:
            return 'Lurker'
    elif 'few times per week' in visit or 'weekly' in visit:
        if part and 'never' not in part:
            return 'Active'
        else:
            return 'Casual'
    else:
        return 'Inactive'

print("✓ Helper functions defined")

✓ Helper functions defined


In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

DATA_PATH = '../data/processed/cleaned_full.csv'
FIGURES_PATH = '../visualizations/'
TABLES_PATH = '../results/tables/'
RANDOM_STATE = 42
N_CLUSTERS = 4  # Number of platform usage clusters

# Create output directories
for path in [FIGURES_PATH, TABLES_PATH]:
    os.makedirs(path, exist_ok=True)

print("="*80)
print("COMMUNITY PLATFORM EFFECTIVENESS ANALYSIS")
print("="*80)

COMMUNITY PLATFORM EFFECTIVENESS ANALYSIS


## Step 1: Load and Prepare Data

In [5]:
print("\n[1/5] Loading and preparing data...")

try:
    df = pd.read_csv(DATA_PATH, low_memory=False)
    print(f"   ✓ Loaded {len(df):,} responses")
except FileNotFoundError:
    print(f"   ✗ Error: File not found at {DATA_PATH}")
    sys.exit(1)

# Identify relevant columns
so_columns = [col for col in df.columns if any(x in col for x in ['SOVisit', 'SOPart', 'SOComm'])]
# CORRECTED: Use CommPlatform columns for community platforms (Stack Overflow, Discord, Reddit, etc.)
platform_columns = [col for col in df.columns if 'CommPlatform' in col]
experience_columns = [col for col in df.columns if 'YearsCode' in col]

print(f"\n   Found columns:")
print(f"   • SO engagement: {len(so_columns)} columns")
print(f"   • Community platforms: {len(platform_columns)} columns")
print(f"   • Experience: {len(experience_columns)} columns")

# Display sample column names
print(f"\n   SO columns: {so_columns[:5]}")
print(f"   Community Platform columns: {platform_columns}")
print(f"   Experience columns: {experience_columns}")

# Create experience categories
if 'YearsCodePro' in df.columns:
    df['experience_category'] = df['YearsCodePro'].apply(create_experience_category)
    print(f"\n   Experience distribution:")
    print(df['experience_category'].value_counts())
elif 'experience_category' not in df.columns:
    print("   ⚠ Warning: No experience data available")

# Create SO engagement categories
df['so_engagement'] = df.apply(categorize_so_engagement, axis=1)
print(f"\n   SO Engagement distribution:")
print(df['so_engagement'].value_counts())

# Create binary active SO user flag for cohort analysis
df['is_active_so'] = df['so_engagement'].isin(['Very Active', 'Active']).astype(int)
print(f"\n   Active SO users: {df['is_active_so'].sum():,} ({df['is_active_so'].mean()*100:.1f}%)")


[1/5] Loading and preparing data...
   ✓ Loaded 49,123 responses

   Found columns:
   • SO engagement: 3 columns
   • Community platforms: 5 columns
   • Experience: 1 columns

   SO columns: ['SOVisitFreq', 'SOPartFreq', 'SOComm']
   Community Platform columns: ['CommPlatformHaveWorkedWith', 'CommPlatformWantToWorkWith', 'CommPlatformAdmired', 'CommPlatformHaveEntr', 'CommPlatformWantEntr']
   Experience columns: ['YearsCode']

   SO Engagement distribution:
so_engagement
Inactive       22220
Active         13933
Lurker          6977
Casual          5693
Very Active      300
Name: count, dtype: int64

   Active SO users: 14,233 (29.0%)


In [6]:
# Display the community platforms found in the dataset
print("\n" + "="*80)
print("COMMUNITY PLATFORMS IDENTIFIED IN DATASET")
print("="*80)

if 'CommPlatformHaveWorkedWith' in df.columns:
    all_platforms_found = set()
    for val in df['CommPlatformHaveWorkedWith'].dropna():
        if isinstance(val, str):
            platforms = [p.strip() for p in val.split(';')]
            all_platforms_found.update(platforms)
    
    print(f"\nTotal unique platforms: {len(all_platforms_found)}\n")
    for i, platform in enumerate(sorted(all_platforms_found), 1):
        # Count usage
        usage_count = df['CommPlatformHaveWorkedWith'].astype(str).str.contains(platform, case=False, na=False).sum()
        usage_pct = (usage_count / len(df)) * 100
        print(f"  {i:2}. {platform:<40} ({usage_count:,} users, {usage_pct:.1f}%)")
else:
    print("⚠ CommPlatformHaveWorkedWith column not found!")

print("\n" + "="*80)


COMMUNITY PLATFORMS IDENTIFIED IN DATASET

Total unique platforms: 18

   1. Bluesky                                  (3,262 users, 6.6%)
   2. Company-sponsored forums                 (1,859 users, 3.8%)
   3. Dev.to                                   (3,456 users, 7.0%)
   4. Discord                                  (11,751 users, 23.9%)
   5. GitHub (public projects, not private repos) (0 users, 0.0%)
   6. Hacker News                              (5,913 users, 12.0%)
   7. Hashnode                                 (350 users, 0.7%)
   8. Kaggle                                   (1,298 users, 2.6%)
   9. LinkedIn                                 (11,243 users, 22.9%)
  10. Medium                                   (8,845 users, 18.0%)
  11. Reddit                                   (16,223 users, 33.0%)
  12. Slack (public channels, not work)        (0 users, 0.0%)
  13. Stack Exchange                           (14,031 users, 28.6%)
  14. Stack Overflow                           (25,431

## Step 2: Platform Usage Patterns - Cluster Analysis

In [7]:
print("\n[2/5] Performing cluster analysis on community platform usage...")

# Prepare data for clustering
# CORRECTED: Use CommPlatformHaveWorkedWith for community platforms
platform_data = []

if 'CommPlatformHaveWorkedWith' in df.columns:
    # Get the community platform column
    platform_col = 'CommPlatformHaveWorkedWith'
    
    # Extract all unique platforms
    all_platforms = set()
    for val in df[platform_col].dropna():
        if isinstance(val, str):
            platforms = [p.strip() for p in val.split(';')]
            all_platforms.update(platforms)
    
    all_platforms = sorted(list(all_platforms))  # Keep all platforms
    print(f"   ✓ Identified {len(all_platforms)} community platforms:")
    print(f"   Platforms: {', '.join(all_platforms)}")
    
    # Create binary features for each platform
    for platform in all_platforms:
        df[f'uses_{platform}'] = df[platform_col].astype(str).str.contains(platform, case=False, na=False).astype(int)
    
    # Features for clustering
    platform_features = [f'uses_{p}' for p in all_platforms]
    
    # Add SO engagement as features
    df['so_visits_numeric'] = df.get('SOVisitFreq', '').astype(str).map({
        'Multiple times per day': 5,
        'Daily or almost daily': 4,
        'A few times per week': 3,
        'A few times per month or weekly': 2,
        'Less than once per month or monthly': 1,
        'Never': 0
    }).fillna(0)
    
    df['so_part_numeric'] = df.get('SOPartFreq', '').astype(str).map({
        'Multiple times per day': 5,
        'Daily or almost daily': 4,
        'A few times per week': 3,
        'A few times per month or weekly': 2,
        'Less than once per month or monthly': 1,
        'Never': 0
    }).fillna(0)
    
    platform_features.extend(['so_visits_numeric', 'so_part_numeric'])
    
    # Prepare clustering data
    cluster_data = df[platform_features].dropna()
    
    if len(cluster_data) > 0:
        # Standardize features
        scaler = StandardScaler()
        cluster_data_scaled = scaler.fit_transform(cluster_data)
        
        # Perform K-means clustering
        kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
        clusters = kmeans.fit_predict(cluster_data_scaled)
        
        df.loc[cluster_data.index, 'platform_cluster'] = clusters
        
        print(f"   ✓ Created {N_CLUSTERS} platform usage clusters")
        print(f"\n   Cluster distribution:")
        print(df['platform_cluster'].value_counts().sort_index())
        
        # Analyze cluster characteristics - IMPROVED VERSION
        cluster_profiles = []
        detailed_platform_usage = []
        
        for cluster_id in range(N_CLUSTERS):
            cluster_mask = df['platform_cluster'] == cluster_id
            cluster_size = cluster_mask.sum()
            
            # Calculate usage rate for ALL platforms
            platform_usage_rates = []
            for platform in all_platforms:
                usage_rate = df.loc[cluster_mask, f'uses_{platform}'].mean()
                platform_usage_rates.append((platform, usage_rate))
            
            # Sort by usage rate
            platform_usage_rates.sort(key=lambda x: x[1], reverse=True)
            
            # Get top platforms (lowered threshold to 10% to capture more patterns)
            top_platforms = [p for p, rate in platform_usage_rates if rate > 0.10]
            
            # Calculate average number of platforms per user
            platform_cols = [f'uses_{p}' for p in all_platforms]
            avg_platforms = df.loc[cluster_mask, platform_cols].sum(axis=1).mean()
            
            # SO engagement
            avg_visits = df.loc[cluster_mask, 'so_visits_numeric'].mean()
            avg_part = df.loc[cluster_mask, 'so_part_numeric'].mean()
            
            # Create profile with top 5 platforms showing percentages
            top_5_with_pct = [f"{p} ({r*100:.0f}%)" for p, r in platform_usage_rates[:5]]
            
            cluster_profiles.append({
                'Cluster': cluster_id,
                'Size': cluster_size,
                'Avg_Platforms_Per_User': round(avg_platforms, 1),
                'Top_Platforms': ', '.join(top_5_with_pct),
                'Avg_SO_Visits': round(avg_visits, 2),
                'Avg_SO_Participation': round(avg_part, 2)
            })
            
            # Store detailed platform usage for each cluster (for detailed CSV)
            for platform, rate in platform_usage_rates:
                detailed_platform_usage.append({
                    'Cluster': cluster_id,
                    'Platform': platform,
                    'Usage_Rate': round(rate * 100, 2)
                })
        
        cluster_df = pd.DataFrame(cluster_profiles)
        print(f"\n   Cluster profiles:")
        print(cluster_df.to_string(index=False))
        
        # Save summary cluster profiles
        safe_save_csv(cluster_df, f'{TABLES_PATH}platform_clusters.csv')
        
        # Save detailed platform usage per cluster
        detailed_df = pd.DataFrame(detailed_platform_usage)
        detailed_pivot = detailed_df.pivot(index='Platform', columns='Cluster', values='Usage_Rate')
        detailed_pivot.columns = [f'Cluster_{c}_Pct' for c in detailed_pivot.columns]
        safe_save_csv(detailed_pivot, f'{TABLES_PATH}platform_clusters_detailed.csv')
        
        # Visualize clusters - ENHANCED VERSION
        fig = plt.figure(figsize=(18, 10))
        gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)
        
        # 1. Cluster sizes
        ax1 = fig.add_subplot(gs[0, 0])
        cluster_counts = df['platform_cluster'].value_counts().sort_index()
        bars = ax1.bar(cluster_counts.index, cluster_counts.values, color='steelblue', edgecolor='navy', alpha=0.7)
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height):,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
        ax1.set_xlabel('Cluster', fontsize=12, fontweight='bold')
        ax1.set_ylabel('Number of Developers', fontsize=12, fontweight='bold')
        ax1.set_title('Platform Usage Cluster Sizes', fontsize=14, fontweight='bold')
        ax1.grid(axis='y', alpha=0.3)
        
        # 2. SO engagement by cluster
        ax2 = fig.add_subplot(gs[0, 1])
        x = np.arange(len(cluster_df))
        width = 0.35
        ax2.bar(x - width/2, cluster_df['Avg_SO_Visits'], width, label='Visits', color='coral', alpha=0.8)
        ax2.bar(x + width/2, cluster_df['Avg_SO_Participation'], width, label='Participation', color='mediumseagreen', alpha=0.8)
        ax2.set_xlabel('Cluster', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Average Score', fontsize=12, fontweight='bold')
        ax2.set_title('SO Engagement by Platform Cluster', fontsize=14, fontweight='bold')
        ax2.set_xticks(x)
        ax2.set_xticklabels(cluster_df['Cluster'])
        ax2.legend()
        ax2.grid(axis='y', alpha=0.3)
        
        # 3. Platform usage heatmap (top 10 platforms across all clusters)
        ax3 = fig.add_subplot(gs[1, :])
        # Get top 10 most used platforms overall
        overall_platform_usage = []
        for platform in all_platforms:
            avg_usage = df[f'uses_{platform}'].mean()
            overall_platform_usage.append((platform, avg_usage))
        overall_platform_usage.sort(key=lambda x: x[1], reverse=True)
        top_10_platforms = [p for p, _ in overall_platform_usage[:10]]
        
        # Create matrix for heatmap
        heatmap_data = []
        for cluster_id in range(N_CLUSTERS):
            cluster_mask = df['platform_cluster'] == cluster_id
            row = []
            for platform in top_10_platforms:
                usage_rate = df.loc[cluster_mask, f'uses_{platform}'].mean() * 100
                row.append(usage_rate)
            heatmap_data.append(row)
        
        heatmap_df = pd.DataFrame(heatmap_data, 
                                  index=[f'Cluster {i}' for i in range(N_CLUSTERS)],
                                  columns=top_10_platforms)
        
        sns.heatmap(heatmap_df, annot=True, fmt='.1f', cmap='YlOrRd', 
                   cbar_kws={'label': 'Usage Rate (%)'}, ax=ax3,
                   linewidths=0.5, linecolor='white')
        ax3.set_title('Top 10 Platform Usage Rates by Cluster (%)', fontsize=14, fontweight='bold')
        ax3.set_xlabel('Community Platform', fontsize=12, fontweight='bold')
        ax3.set_ylabel('Cluster', fontsize=12, fontweight='bold')
        
        plt.savefig(f'{FIGURES_PATH}platform_clusters.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("   ✓ Saved: platform_clusters.png (enhanced version with heatmap)")
    else:
        print("   ⚠ Insufficient data for clustering")
else:
    print("   ⚠ No learning platform columns found")


[2/5] Performing cluster analysis on community platform usage...
   ✓ Identified 18 community platforms:
   Platforms: Bluesky, Company-sponsored forums, Dev.to, Discord, GitHub (public projects, not private repos), Hacker News, Hashnode, Kaggle, LinkedIn, Medium, Reddit, Slack (public channels, not work), Stack Exchange, Stack Overflow, Substack, Twitch, X, YouTube
   ✓ Created 4 platform usage clusters

   Cluster distribution:
platform_cluster
0.0     4319
1.0    25223
2.0    19231
3.0      350
Name: count, dtype: int64

   Cluster profiles:
 Cluster  Size  Avg_Platforms_Per_User                                                                    Top_Platforms  Avg_SO_Visits  Avg_SO_Participation
       0  4319                     7.7        YouTube (88%), Stack Overflow (87%), Reddit (76%), X (72%), Discord (69%)           2.53                  0.27
       1 25223                     0.4     Stack Overflow (13%), YouTube (7%), Reddit (5%), Discord (5%), LinkedIn (4%)           0.70

## Step 3: Cohort Analysis by Experience Level

In [8]:
print("\n[3/5] Performing cohort analysis by experience level...")

if 'experience_category' in df.columns:
    # Platform usage by experience
    cohort_analysis = []
    
    # Get actual experience levels from the data
    exp_levels = df['experience_category'].dropna().unique()
    exp_levels = sorted([e for e in exp_levels if e != 'Unknown'])  # Sort and remove Unknown
    
    for exp_level in exp_levels:
        cohort = df[df['experience_category'] == exp_level]
        
        if len(cohort) > 0:
            # Safely calculate active rate
            if 'is_active_so' in cohort.columns:
                active_rate = (cohort['is_active_so'].sum() / len(cohort) * 100)
            else:
                active_rate = 0
            
            cohort_stats = {
                'Experience': exp_level,
                'Count': len(cohort),
                'Avg_SO_Visits': cohort['so_visits_numeric'].mean(),
                'Avg_SO_Participation': cohort['so_part_numeric'].mean(),
                'Active_Rate': active_rate,
                'Very_Active_Rate': (cohort['so_engagement'] == 'Very Active').sum() / len(cohort) * 100
            }
            cohort_analysis.append(cohort_stats)
    
    cohort_df = pd.DataFrame(cohort_analysis)
    
    print(f"\n   Cohort analysis results:")
    print(cohort_df.to_string(index=False))
    
    # Save cohort analysis
    safe_save_csv(cohort_df, f'{TABLES_PATH}experience_cohort_analysis.csv')
    
    # Visualize cohort patterns
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # SO Visit frequency by experience
    axes[0, 0].bar(range(len(cohort_df)), cohort_df['Avg_SO_Visits'], color='steelblue')
    axes[0, 0].set_xticks(range(len(cohort_df)))
    axes[0, 0].set_xticklabels([e.split('(')[0].strip() for e in cohort_df['Experience']], rotation=45)
    axes[0, 0].set_ylabel('Average Visit Score')
    axes[0, 0].set_title('SO Visit Frequency by Experience')
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    # SO Participation by experience
    axes[0, 1].bar(range(len(cohort_df)), cohort_df['Avg_SO_Participation'], color='coral')
    axes[0, 1].set_xticks(range(len(cohort_df)))
    axes[0, 1].set_xticklabels([e.split('(')[0].strip() for e in cohort_df['Experience']], rotation=45)
    axes[0, 1].set_ylabel('Average Participation Score')
    axes[0, 1].set_title('SO Participation by Experience')
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    # Active user rate
    axes[1, 0].bar(range(len(cohort_df)), cohort_df['Active_Rate'], color='mediumseagreen')
    axes[1, 0].set_xticks(range(len(cohort_df)))
    axes[1, 0].set_xticklabels([e.split('(')[0].strip() for e in cohort_df['Experience']], rotation=45)
    axes[1, 0].set_ylabel('Active User Rate (%)')
    axes[1, 0].set_title('Active SO Users by Experience')
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    # Cohort sizes
    axes[1, 1].bar(range(len(cohort_df)), cohort_df['Count'], color='mediumpurple')
    axes[1, 1].set_xticks(range(len(cohort_df)))
    axes[1, 1].set_xticklabels([e.split('(')[0].strip() for e in cohort_df['Experience']], rotation=45)
    axes[1, 1].set_ylabel('Number of Developers')
    axes[1, 1].set_title('Cohort Sizes')
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{FIGURES_PATH}experience_cohort_analysis.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("   ✓ Saved: experience_cohort_analysis.png")
    
    # Statistical tests
    print(f"\n   Statistical significance tests:")
    
    # Chi-square test for experience vs SO engagement
    contingency = pd.crosstab(df['experience_category'], df['so_engagement'])
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    
    print(f"   • Experience vs SO Engagement:")
    print(f"     χ² = {chi2:.2f}, p-value = {p_value:.4e}")
    if p_value < 0.05:
        print(f"     ✓ Significant relationship detected")
    else:
        print(f"     ✗ No significant relationship")
else:
    print("   ⚠ Experience category not available")


[3/5] Performing cohort analysis by experience level...

   Cohort analysis results:
   Experience  Count  Avg_SO_Visits  Avg_SO_Participation  Active_Rate  Very_Active_Rate
 Expert (10+)  25800       2.083488              0.304845    39.686047          0.872093
 Junior (0-2)   1705       1.122581              0.143695    10.557185          0.410557
    Mid (3-5)   5146       1.643995              0.124563    17.897396          0.330354
Senior (6-10)  10349       1.973524              0.174703    26.901150          0.454150
   ✓ Saved to: ./tables/experience_cohort_analysis.csv
   ✓ Saved: experience_cohort_analysis.png

   Statistical significance tests:
   • Experience vs SO Engagement:
     χ² = 2260.45, p-value = 0.0000e+00
     ✓ Significant relationship detected


## Step 4: Platform Cluster vs SO Engagement Analysis

In [9]:
print("\n[4/5] Analyzing platform cluster patterns vs SO engagement...")

if 'platform_cluster' in df.columns:
    # Create crosstab for cluster vs engagement
    cluster_engagement = pd.crosstab(
        df['platform_cluster'], 
        df['so_engagement'],
        normalize='index'
    ) * 100  # Convert to percentages
    
    # Rename index for clarity
    cluster_engagement.index = [f'Cluster {i}' for i in cluster_engagement.index]
    
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Heatmap showing percentage distribution
    sns.heatmap(
        cluster_engagement, 
        annot=True, 
        fmt='.1f', 
        cmap='YlOrRd',
        cbar_kws={'label': 'Percentage (%)'},
        ax=axes[0]
    )
    axes[0].set_title('Platform Cluster Engagement Patterns (%)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('SO Engagement Level', fontsize=12)
    axes[0].set_ylabel('Platform Usage Cluster', fontsize=12)
    
    # Grouped bar chart showing counts
    cluster_engagement_counts = pd.crosstab(df['platform_cluster'], df['so_engagement'])
    cluster_engagement_counts.index = [f'Cluster {i}' for i in cluster_engagement_counts.index]
    
    cluster_engagement_counts.plot(kind='bar', ax=axes[1], width=0.8)
    axes[1].set_title('Engagement Distribution by Platform Cluster', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Platform Usage Cluster', fontsize=12)
    axes[1].set_ylabel('Number of Developers', fontsize=12)
    axes[1].legend(title='SO Engagement', bbox_to_anchor=(1.05, 1), loc='upper left')
    axes[1].grid(axis='y', alpha=0.3)
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
    
    plt.tight_layout()
    plt.savefig(f'{FIGURES_PATH}cluster_engagement_analysis.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("   ✓ Saved: cluster_engagement_analysis.png")
    
    # Save the percentage table
    safe_save_csv(cluster_engagement.round(1), f'{TABLES_PATH}cluster_engagement_distribution.csv')
    
    # Print key insights
    print(f"\n   Key Patterns:")
    for cluster in cluster_engagement.index:
        most_common = cluster_engagement.loc[cluster].idxmax()
        percentage = cluster_engagement.loc[cluster].max()
        print(f"   • {cluster}: {percentage:.1f}% are '{most_common}'")
    
else:
    print("   ⚠ Platform cluster data not available")


[4/5] Analyzing platform cluster patterns vs SO engagement...
   ✓ Saved: cluster_engagement_analysis.png
   ✓ Saved to: ./tables/cluster_engagement_distribution.csv

   Key Patterns:
   • Cluster 0.0: 42.1% are 'Active'
   • Cluster 1.0: 76.9% are 'Inactive'
   • Cluster 2.0: 45.5% are 'Active'
   • Cluster 3.0: 39.7% are 'Active'


## Step 5: Summary and Key Insights

In [10]:
print("\n[5/5] Generating summary report...")
print("\n" + "="*80)
print("ANALYSIS COMPLETE - KEY INSIGHTS")
print("="*80)

print("\n📊 Platform Usage Patterns:")
if 'platform_cluster' in df.columns:
    print(f"   • Identified {N_CLUSTERS} distinct platform usage clusters")
    print(f"   • Clusters show varying levels of SO engagement")
    print(f"   • Platform diversity correlates with engagement levels")
else:
    print("   • Clustering analysis not available")

print("\n👥 Cohort Insights:")
if 'cohort_df' in locals() and len(cohort_df) > 0:
    most_active = cohort_df.loc[cohort_df['Active_Rate'].idxmax()]
    print(f"   • Most active cohort: {most_active['Experience']}")
    print(f"   • Active rate: {most_active['Active_Rate']:.1f}%")
    print(f"   • Experience level significantly affects engagement (p < 0.05)")
else:
    print("   • Cohort analysis not available")

print("\n💡 Recommendations:")
print("   1. Target platform-specific outreach to high-engagement clusters")
print("   2. Create experience-appropriate onboarding paths")
print("   3. Foster community connections early in developer journey")
print("   4. Leverage popular community platforms for developer engagement")

print("\n✅ All visualizations saved to:", FIGURES_PATH)
print("✅ All tables saved to:", TABLES_PATH)
print("="*80)

# Summary statistics
summary_stats = {
    'Total Respondents': len(df),
    'Platform Clusters': N_CLUSTERS if 'platform_cluster' in df else 'N/A',
    'Experience Cohorts': len(cohort_df) if 'cohort_df' in locals() else 'N/A',
    'Avg Platforms Per User': round(df[[f'uses_{p}' for p in all_platforms]].sum(axis=1).mean(), 2) if 'all_platforms' in locals() else 'N/A'
}

summary_df = pd.DataFrame([summary_stats])
safe_save_csv(summary_df, f'{TABLES_PATH}analysis_summary.csv')

print("\n📈 Analysis Summary:")
for key, value in summary_stats.items():
    print(f"   • {key}: {value}")


[5/5] Generating summary report...

ANALYSIS COMPLETE - KEY INSIGHTS

📊 Platform Usage Patterns:
   • Identified 4 distinct platform usage clusters
   • Clusters show varying levels of SO engagement
   • Platform diversity correlates with engagement levels

👥 Cohort Insights:
   • Most active cohort: Expert (10+)
   • Active rate: 39.7%
   • Experience level significantly affects engagement (p < 0.05)

💡 Recommendations:
   1. Target platform-specific outreach to high-engagement clusters
   2. Create experience-appropriate onboarding paths
   3. Foster community connections early in developer journey
   4. Leverage popular community platforms for developer engagement

✅ All visualizations saved to: ./visualizations/
✅ All tables saved to: ./tables/
   ✓ Saved to: ./tables/analysis_summary.csv

📈 Analysis Summary:
   • Total Respondents: 49123
   • Platform Clusters: 4
   • Experience Cohorts: 4
   • Avg Platforms Per User: 2.92
